<a href="https://colab.research.google.com/github/irullah/comic-yolo/blob/main/Auto_Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
import zipfile
import glob
from ultralytics import YOLO
from google.colab import files

# ==========================================
# 1. SETUP FOLDER
# ==========================================
input_dir = "dataset_mentah"
output_dir = "siap_roboflow"
yolo_runs_dir = "runs/detect"  # Default folder YOLOv8 untuk hasil predict

if os.path.exists(input_dir): shutil.rmtree(input_dir)
if os.path.exists(output_dir): shutil.rmtree(output_dir)
if os.path.exists("runs"): shutil.rmtree("runs") # Bersihkan sisa prediksi lama

os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# 2. EKSTRAK ZIP GAMBAR MENTAH
# ==========================================
zip_candidates = [f for f in os.listdir('.') if f.lower().endswith('.zip') and not f.startswith('S')]

if not zip_candidates:
    print("❌ ERROR: Upload dulu file ZIP berisi gambar mentah ke panel kiri Colab!")
else:
    zip_path = zip_candidates[0]
    print(f"📦 Mengekstrak {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(input_dir)

    # ==========================================
    # 3. JALANKAN PREDIKSI YOLO (SAVE TXT)
    # ==========================================
    print("🤖 Memulai Auto-Annotation...")
    model_path = "/content/drive/MyDrive/Colab Notebooks/Comic Cleaner /best.pt"
    model = YOLO(model_path)

    # YOLO hanya akan membuat file .txt, gambar aslinya tidak ditimpa/digambar ulang
    model.predict(
        source=input_dir,
        conf=0.10,        # Nilai rendah agar model menandai semua teks (meski samar)
        save_txt=True,    # INI KUNCI UTAMANYA: Simpan koordinat ke .txt
        save_conf=False,  # Roboflow tidak butuh nilai akurasi
        save=False,       # Jangan simpan gambar hasil visualisasi
        verbose=False
    )

    # ==========================================
    # 4. GABUNGKAN GAMBAR & TXT UNTUK ROBOFLOW
    # ==========================================
    print("📂 Menyatukan gambar dan file label .txt...")

    # Pindahkan semua gambar dari input ke folder output
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
        for img_file in glob.glob(f"{input_dir}/**/{ext}", recursive=True):
            shutil.copy(img_file, output_dir)

    # Pindahkan semua file .txt dari folder YOLO runs ke folder output
    txt_files = glob.glob(f"{yolo_runs_dir}/**/labels/*.txt", recursive=True)
    for txt_file in txt_files:
        shutil.copy(txt_file, output_dir)

    # ==========================================
    # 5. ZIP & DOWNLOAD
    # ==========================================
    output_zip = "Siap_Upload_Roboflow"
    shutil.make_archive(output_zip, 'zip', output_dir)

    print(f"🎉 Selesai! Mengunduh {output_zip}.zip...")
    files.download(f"{output_zip}.zip")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
❌ ERROR: Upload dulu file ZIP berisi gambar mentah ke panel kiri Colab!
